# ===============================================================
# 📜 CSE-475: Semi-Supervised Learning (SSL) with DINO + YOLO
# ===============================================================
# **SylishBD Fish Detection Dataset**

This notebook implements a **DINOv2-style Self-Supervised Learning (SSL)**
pretraining on the **YOLOv10 backbone**, followed by **YOLO12 detector** 
fine-tuning on the **SylishBD fish detection dataset**.

## 🔹 What's Inside:
   1. **COCO → YOLO conversion** (images/labels + data.yaml)
   2. **SSL pretraining** (DINOv2-style):
       - Student/Teacher (EMA) with cosine momentum schedule
       - Multi-crop views: 2 global + 8 local crops
       - Temperature schedule for teacher + probability centering
       - Cross-entropy between teacher probs (global) and student logits (all views)
   3. **Save SSL-pretrained YOLOv10 backbone weights**
   4. **Fine-tune YOLO12 detector** with SSL backbone initialization
   5. **Evaluate** (mP, mR, mAP@0.50, mAP@0.50–0.95)
   6. **Visualize** backbone features (PCA) and detection predictions

## ⚠️ Dataset: SylishBD
   - **Images**: `/kaggle/input/syfish-bd/Sylfish_bd/images`
   - **Annotations**: `/kaggle/input/syfish-bd/Sylfish_bd/annotations` (COCO JSON)
   - **Masks**: `/kaggle/input/syfish-bd/Sylfish_bd/masks` (optional)
   - **Fish species**: Boal, Ilish, Kalibaush, Katla, Koi, Mrigel, Pabda, Rui, Telapia

## ⚠️ Note:
   - This is a **DINOv2-style adaptation** to convolutional backbones for YOLOv10 compatibility
   - For better transfer learning, increase **SSL_EPOCHS** to 50–100
   - Kaggle-compatible: uses `/kaggle/working/` for outputs

---

In [ ]:
!pip -q install torch==2.2.1 torchvision==0.17.1 pytorch-lightning==2.2.1 lightly==1.5.22 ultralytics==8.2.6
!pip -q install opencv-python matplotlib tqdm pycocotools scikit-learn pillow numpy pandas

# ===============================================================
# 0) Setup: Install Required Packages (Kaggle Python 3.11)
# ===============================================================

In [ ]:
print("\n=== FEATURE SPACE VISUALIZATION (PCA) ===")

# Load SSL backbone for feature extraction
det_pca = YOLO("yolov10s.yaml")
det_pca.model.model[0].load_state_dict(
    torch.load(SSL_BACKBONE_W, map_location="cpu"), 
    strict=False
)
bb_pca = det_pca.model.model[0].to(device).eval()

class BBEnc(nn.Module):
    """Backbone encoder for feature extraction."""
    def __init__(self, bb):
        super().__init__()
        self.bb = bb
    
    def forward(self, x):
        out = self.bb(x)
        t = _pick_last_feat(out)
        if t.ndim == 4:
            z = t.mean(dim=(2, 3))
        elif t.ndim == 3:
            z = t.mean(dim=2)
        elif t.ndim == 2:
            z = t
        else:
            z = t.view(t.size(0), -1)
        return z

enc_pca = BBEnc(bb_pca).to(device)

# Transform for feature extraction
tfm_pca = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Sample images from dataset
img_dirs = [YOLO_SPLIT / "train" / "images", YOLO_SPLIT / "val" / "images"]
files = []
for d in img_dirs:
    if d.exists():
        files += list(d.glob("*.*"))

random.shuffle(files)
files = files[:500] if len(files) > 500 else files

print(f"Extracting features from {len(files)} images...")

# Extract features
feats = []
with torch.no_grad():
    for p in tqdm(files, desc="Feature extraction", leave=False):
        try:
            img = Image.open(p).convert("RGB")
            x = tfm_pca(img).unsqueeze(0).to(device)
            z = enc_pca(x)
            feats.append(z.squeeze(0).cpu().numpy())
        except:
            continue

if feats:
    feats = np.stack(feats, axis=0)
    print(f"✓ Feature matrix shape: {feats.shape}")
    
    # PCA
    pca = PCA(n_components=2, random_state=42)
    xy = pca.fit_transform(feats)
    
    # KMeans clustering
    n_clusters = 5
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(feats)
    
    # Visualize
    plt.figure(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
    
    for i in range(n_clusters):
        mask = (labels == i)
        plt.scatter(xy[mask, 0], xy[mask, 1], s=50, label=f"Cluster {i}", 
                   c=[colors[i]], alpha=0.7, edgecolors='black', linewidth=0.5)
    
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%})", fontsize=11)
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%})", fontsize=11)
    plt.title("PCA of SSL-Pretrained YOLOv10 Features\n(SylishBD Dataset)", fontsize=12)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "04_pca_features.png", dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✓ PCA visualization saved: {RESULTS_DIR / '04_pca_features.png'}")
    print(f"  Variance explained by PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  Variance explained by PC2: {pca.explained_variance_ratio_[1]:.2%}")
else:
    print("⚠ No features extracted")

# ===============================================================
# 13) PCA Visualization of SSL-Pretrained Features
# ===============================================================

In [ ]:
print("\n=== DETECTION VISUALIZATION ===")

if best_model_path.exists():
    model_vis = YOLO(str(best_model_path))
    
    # Get test images
    test_img_dir = YOLO_SPLIT / "test" / "images"
    test_images = list(test_img_dir.glob("*.*"))
    
    if test_images:
        # Visualize predictions on random test images
        n_vis = min(6, len(test_images))
        sample_imgs = random.sample(test_images, n_vis)
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(sample_imgs):
            pred = model_vis.predict(
                source=str(img_path),
                imgsz=YOLO_IMG_SIZE,
                conf=0.25,
                device=YOLO_DEVICE,
                verbose=False
            )[0]
            
            # Get annotated image (BGR -> RGB)
            pred_img = pred.plot()[:, :, ::-1]
            
            axes[idx].imshow(pred_img)
            axes[idx].set_title(f"Detections: {len(pred.boxes)}", fontsize=10)
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "03_yolo12_predictions.png", dpi=100, bbox_inches='tight')
        plt.show()
        
        print(f"✓ Visualization saved: {RESULTS_DIR / '03_yolo12_predictions.png'}")
    else:
        print("⚠ No test images found for visualization")
else:
    print("⚠ Best model not found")

# ===============================================================
# 12) Visualization of Detection Results
# ===============================================================

In [ ]:
print("\n=== MODEL EVALUATION ===")

# Find best model
best_model_path = RESULTS_DIR / "yolo12_ssl_sylfishbd" / "weights" / "best.pt"

if best_model_path.exists():
    model_det = YOLO(str(best_model_path))
    
    # Validate on validation set
    print(f"🔄 Evaluating on validation set...")
    val_results = model_det.val(
        data=str(DATA_YAML),
        imgsz=YOLO_IMG_SIZE,
        batch=4,
        device=YOLO_DEVICE,
        verbose=False
    )
    
    # Extract metrics
    try:
        mp = float(val_results.box.mp)
        mr = float(val_results.box.mr)
        map50 = float(val_results.box.map50)
        map5095 = float(val_results.box.map)
    except:
        mp, mr, map50, map5095 = 0, 0, 0, 0
    
    print(f"\n📊 VALIDATION METRICS:")
    print(f"   Precision (mP)  : {mp:.4f}")
    print(f"   Recall (mR)     : {mr:.4f}")
    print(f"   mAP@0.50        : {map50:.4f}")
    print(f"   mAP@0.50-0.95   : {map5095:.4f}")
    
    # Save metrics
    metrics_dict = {
        "precision": float(mp),
        "recall": float(mr),
        "mAP_50": float(map50),
        "mAP_50_95": float(map5095)
    }
    
    with open(RESULTS_DIR / "evaluation_metrics.json", 'w') as f:
        json.dump(metrics_dict, f, indent=2)
    
    print(f"\n✓ Metrics saved to: {RESULTS_DIR / 'evaluation_metrics.json'}")
else:
    print("⚠ Best model not found!")

# ===============================================================
# 11) Model Evaluation and Metrics
# ===============================================================

In [ ]:
print("\n=== YOLO12 DETECTOR FINE-TUNING ===")

# Load SSL-pretrained backbone
det = YOLO("yolov12m.yaml")  # Using YOLOv12m for better performance
missing, unexpected = det.model.model[0].load_state_dict(
    torch.load(SSL_BACKBONE_W, map_location="cpu"), 
    strict=False
)
print(f"✓ SSL backbone loaded")
print(f"  Missing keys: {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

# Training parameters
YOLO_EPOCHS = 10
YOLO_BATCH = 8
YOLO_IMG_SIZE = 640
YOLO_DEVICE = 0 if device == "cuda" else "cpu"

print(f"\n🔄 Starting YOLO12 fine-tuning...")
print(f"   Epochs: {YOLO_EPOCHS}")
print(f"   Batch size: {YOLO_BATCH}")
print(f"   Image size: {YOLO_IMG_SIZE}")
print(f"   Dataset: {DATA_YAML}")

# Train detector
results = det.train(
    data=str(DATA_YAML),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMG_SIZE,
    batch=YOLO_BATCH,
    project=str(RESULTS_DIR),
    name="yolo12_ssl_sylfishbd",
    device=YOLO_DEVICE,
    patience=3,
    save=True,
    verbose=True
)

print(f"\n✓ YOLO12 training complete!")
print(f"  Results saved to: {RESULTS_DIR / 'yolo12_ssl_sylfishbd'}")

# ===============================================================
# 10) YOLO12 Detector Fine-tuning (with SSL backbone)
# ===============================================================

In [ ]:
def coco2yolo(bbox, img_w, img_h):
    """Convert COCO bbox format to YOLO format.
    COCO: [x_min, y_min, width, height]
    YOLO: [x_center, y_center, width, height] (normalized)
    """
    x_min, y_min, bbox_w, bbox_h = bbox
    x_center = (x_min + bbox_w / 2) / img_w
    y_center = (y_min + bbox_h / 2) / img_h
    w_norm = bbox_w / img_w
    h_norm = bbox_h / img_h
    return (x_center, y_center, w_norm, h_norm)


def convert_coco_to_yolo(split_name, img_dir, ann_json_path, out_base):
    """Convert COCO annotations to YOLO format."""
    out_im = out_base / split_name / "images"
    out_lb = out_base / split_name / "labels"
    out_im.mkdir(parents=True, exist_ok=True)
    out_lb.mkdir(parents=True, exist_ok=True)
    
    with open(ann_json_path, 'r') as f:
        coco_data = json.load(f)
    
    # Map image IDs to image metadata
    id_to_img = {}
    for img in coco_data.get("images", []):
        id_to_img[img["id"]] = {
            "id": img["id"],
            "file_name": img["file_name"],
            "width": img["width"],
            "height": img["height"]
        }
    
    # Process annotations
    annotation_count = 0
    image_count = 0
    for ann in coco_data.get("annotations", []):
        img_id = ann["image_id"]
        if img_id not in id_to_img:
            continue
        
        img = id_to_img[img_id]
        yolo_bbox = coco2yolo(ann["bbox"], img["width"], img["height"])
        
        label_file = out_lb / f"{Path(img['file_name']).stem}.txt"
        with open(label_file, 'a') as f:
            class_id = ann.get("category_id", 1) - 1  # 0-indexed
            f.write(f"{class_id} " + " ".join(f"{v:.6f}" for v in yolo_bbox) + "\n")
        
        annotation_count += 1
    
    # Copy images
    for img_info in id_to_img.values():
        src = Path(img_dir) / img_info["file_name"]
        dst = out_im / img_info["file_name"]
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            image_count += 1
    
    return image_count, annotation_count


print("\n=== CONVERTING COCO ANNOTATIONS TO YOLO FORMAT ===")

if not DATA_YAML.exists():
    # Find annotation files (per species)
    ann_files = list(ANNOTATIONS_DIR.glob("*.json"))
    
    if ann_files:
        # Use first annotation file as reference (usually has all categories)
        ref_ann = ann_files[0]
        print(f"Using annotation file: {ref_ann.name}")
        
        with open(ref_ann, 'r') as f:
            ref_data = json.load(f)
        
        # Extract class names from categories
        categories = ref_data.get("categories", [])
        if isinstance(categories, dict):
            # If categories is a dict
            names = [categories.get(str(i), f"class_{i}") for i in range(len(categories))]
        else:
            # If categories is a list
            names = sorted([c.get("name", f"class_{c.get('id', 0)}") 
                          for c in categories], 
                          key=lambda x: x)
        
        print(f"Classes found: {len(names)}")
        print(f"  {names}")
        
        # Convert annotations - create train/val/test splits
        # For now, use all data for training (can adjust for multi-file annotations)
        total_imgs = 0
        total_anns = 0
        
        for ann_file in ann_files[:1]:  # Use first annotation file
            species = ann_file.stem
            img_dir = IMAGES_DIR / species
            if img_dir.exists():
                n_img, n_ann = convert_coco_to_yolo("train", img_dir, ann_file, YOLO_SPLIT)
                total_imgs += n_img
                total_anns += n_ann
        
        # Duplicate train split for val/test (can be improved with actual splits)
        for split in ["val", "test"]:
            split_dir = YOLO_SPLIT / split
            train_dir = YOLO_SPLIT / "train"
            if train_dir.exists() and not (split_dir / "images").exists():
                split_dir.mkdir(parents=True, exist_ok=True)
                (split_dir / "images").mkdir(exist_ok=True)
                (split_dir / "labels").mkdir(exist_ok=True)
                
                # Copy a subset to val/test
                train_imgs = list((train_dir / "images").glob("*.*"))
                sample_size = max(1, len(train_imgs) // 4)
                for img in random.sample(train_imgs, min(sample_size, len(train_imgs))):
                    shutil.copy2(img, split_dir / "images" / img.name)
                    lbl = train_dir / "labels" / f"{img.stem}.txt"
                    if lbl.exists():
                        shutil.copy2(lbl, split_dir / "labels" / lbl.name)
        
        # Create data.yaml for YOLO
        data_dict = {
            "path": str(YOLO_SPLIT),
            "train": "train/images",
            "val": "val/images",
            "test": "test/images",
            "nc": len(names),
            "names": {i: name for i, name in enumerate(names)}
        }
        
        DATA_YAML.write_text(yaml.dump(data_dict, default_flow_style=False))
        
        print(f"\n✓ YOLO conversion complete!")
        print(f"  Total images copied: {total_imgs}")
        print(f"  Total annotations: {total_anns}")
        print(f"  YOLO config: {DATA_YAML}")
    else:
        print("⚠ No annotation files found!")
else:
    print("✓ YOLO dataset already prepared")

# ===============================================================
# 9) Convert COCO Annotations to YOLO Format
# ===============================================================

In [ ]:
# ========== SSL TRAINING PARAMETERS ==========
EPOCHS = 5  # Increase to 50–100 for stronger SSL pretraining
BATCH = 16
ACCUM = 2
LR = 5e-4
WD = 0.05
EMA0 = 0.996
N_LOCAL = 8
NUM_WORKERS = 0

print("\n=== SSL PRETRAINING PARAMETERS ===")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH}")
print(f"Learning rate: {LR}")
print(f"Weight decay: {WD}")
print(f"EMA momentum base: {EMA0}")
print(f"Local crops: {N_LOCAL}")

# Check if SSL backbone exists (skip if found)
if SSL_BACKBONE_W.exists():
    print(f"\n✓ SSL backbone cache found → Skipping pretraining")
    print(f"  Loading from: {SSL_BACKBONE_W}")
else:
    print(f"\n🔄 Starting DINOv2-style SSL pretraining on SylishBD ...")
    
    # Create dataset and dataloader
    ds = MultiCropDINO([UNLABELED_DIR], g_size=224, l_size=96, n_local=N_LOCAL)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=True, drop_last=True)
    print(f"   Dataset: {len(ds)} images")
    print(f"   Batches per epoch: {len(dl)}")
    
    # Optimizer and scheduler
    params = list(student_enc.parameters()) + list(student_head.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
    lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    
    # Loss function
    loss_fn = DINOLoss(OUT_DIM, n_global=2,
                       teacher_temp_warm=0.04, teacher_temp=0.07,
                       warmup_frac=0.1, total_epochs=EPOCHS,
                       center_m=0.9, student_temp=0.1).to(device)
    
    steps_per_epoch = len(dl)
    total_steps = steps_per_epoch * EPOCHS
    gstep = 0
    
    # Training loop
    student_enc.train()
    student_head.train()
    ssl_losses = []
    
    for ep in range(EPOCHS):
        ep_loss = 0.0
        accum = 0
        opt.zero_grad(set_to_none=True)
        
        for views in tqdm(dl, desc=f"SSL Ep {ep+1}/{EPOCHS}", leave=False):
            views = [v.to(device) for v in views]  # [g1, g2, l1..lN]
            
            with autocast_ctx():
                # Student forward (all crops)
                s_feats = [student_enc(v) for v in views]
                s_outs = [student_head(z) for z in s_feats]
                
                # Teacher forward (globals only)
                with torch.no_grad():
                    t_feats = [teacher_enc(v) for v in views[:2]]
                    t_outs = [teacher_head(z) for z in t_feats]
                
                loss = loss_fn(s_outs, t_outs, epoch=ep) / ACCUM
            
            scaler.scale(loss).backward()
            accum += 1
            
            if accum == ACCUM:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(params, 3.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                accum = 0
                
                # EMA update (cosine momentum schedule)
                m = 1 - (1 - EMA0) * (math.cos(math.pi * gstep / total_steps) + 1) / 2
                with torch.no_grad():
                    for ps, pt in zip(student_bb.parameters(), teacher_bb.parameters()):
                        pt.data.mul_(m).add_(ps.data, alpha=1 - m)
                    for ps, pt in zip(student_head.parameters(), teacher_head.parameters()):
                        pt.data.mul_(m).add_(ps.data, alpha=1 - m)
                gstep += 1
            
            ep_loss += loss.item() * ACCUM
        
        lr_sched.step()
        ep_loss_avg = ep_loss / steps_per_epoch
        ssl_losses.append(ep_loss_avg)
        print(f"   SSL Epoch {ep+1:02d}/{EPOCHS}: loss={ep_loss_avg:.4f}")
    
    # Save SSL backbone
    SSL_BACKBONE_W.parent.mkdir(parents=True, exist_ok=True)
    torch.save(student_bb.state_dict(), SSL_BACKBONE_W)
    del ds, dl
    gc.collect()
    
    print(f"\n✓ SSL pretraining complete!")
    print(f"  Backbone saved: {SSL_BACKBONE_W}")
    
    # Plot SSL training loss
    if ssl_losses:
        plt.figure(figsize=(8, 4))
        plt.plot(ssl_losses, marker='o', label='SSL Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('DINO SSL Pretraining Loss (SylishBD)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "02_ssl_training_loss.png", dpi=100, bbox_inches='tight')
        plt.show()

# ===============================================================
# 8) DINOv2-style SSL Pretraining
# ===============================================================

**Note:** For better SSL performance, increase `SSL_EPOCHS` to 50–100.
Set in Kaggle: Use "Factory Reset" to clear cache if changing epochs.

In [ ]:
OUT_DIM = 256

def projector(in_dim, hid=2048, out=OUT_DIM):
    """Create a projector head (linear + GELU + BatchNorm + linear)."""
    return nn.Sequential(
        nn.Linear(in_dim, hid),
        nn.GELU(),
        nn.BatchNorm1d(hid),
        nn.Linear(hid, out)
    )

# Create student and teacher projection heads
student_head = projector(feat_dim).to(device)
teacher_head = projector(feat_dim).to(device)
teacher_head.load_state_dict(student_head.state_dict())
for p in teacher_head.parameters():
    p.requires_grad = False

print(f"✓ Student head and teacher head created (dim: {OUT_DIM})")


class DINOLoss(nn.Module):
    """
    DINO loss: cross-entropy between teacher probabilities (global crops)
    and student logits (all crops), with momentum centering.
    """
    def __init__(self, out_dim, n_global=2,
                 teacher_temp_warm=0.04, teacher_temp=0.07,
                 warmup_frac=0.1, total_epochs=100,
                 center_m=0.9, student_temp=0.1):
        super().__init__()
        self.n_global = n_global
        self.t_warm, self.t_final = teacher_temp_warm, teacher_temp
        self.warmup_frac, self.T = warmup_frac, total_epochs
        self.cm = center_m
        self.student_temp = student_temp
        self.register_buffer("center", torch.zeros(1, out_dim))
    
    def teacher_T(self, epoch):
        """Cosine annealing schedule for teacher temperature."""
        if epoch < self.warmup_frac * self.T:
            a = epoch / (self.warmup_frac * self.T)
            return self.t_warm + (self.t_final - self.t_warm) * a
        return self.t_final
    
    def forward(self, s_out, t_out, epoch):
        Tt = self.teacher_T(epoch)
        
        # Student: logits (scaled by temperature)
        s_logits = [s / self.student_temp for s in s_out]
        
        # Teacher: probabilities (with centering)
        t_probs = [F.softmax((t - self.center) / Tt, dim=-1).detach() for t in t_out]
        
        # Cross-entropy
        loss = 0
        cnt = 0
        for iq, q in enumerate(t_probs):
            for v, s in enumerate(s_logits):
                if v == iq:  # skip same-index global view
                    continue
                loss += -(q * F.log_softmax(s, dim=-1)).sum(1).mean()
                cnt += 1
        loss /= max(cnt, 1)
        
        # Update center (momentum)
        with torch.no_grad():
            batch_center = torch.cat(t_out, 0).mean(0, keepdim=True)
            self.center = self.center * self.cm + (1 - self.cm) * batch_center
        
        return loss

print("✓ DINO loss function created")

# ===============================================================
# 7) DINOv2-style Projector Heads and Loss Function
# ===============================================================

In [ ]:
print("\n=== BUILDING YOLOV10 BACKBONE ===")

# Load YOLOv10s model
detector = YOLO("yolov10s.yaml")
model = detector.model

# Extract backbone (student and teacher)
student_bb = model.model[0].to(device)
teacher_bb = YOLO("yolov10s.yaml").model.model[0].to(device)
teacher_bb.load_state_dict(student_bb.state_dict())
for p in teacher_bb.parameters():
    p.requires_grad = False

print(f"✓ Student backbone: {student_bb.__class__.__name__}")
print(f"✓ Teacher backbone: {teacher_bb.__class__.__name__} (frozen)")


def _pick_last_feat(out):
    """
    Extract deepest feature map/vector from backbone output.
    Handles various output formats (list, dict, tensor).
    """
    if isinstance(out, (list, tuple)) and len(out) > 0:
        t = out[-1]
    elif isinstance(out, dict) and len(out) > 0:
        t = list(out.values())[-1]
    else:
        t = out
    
    if not torch.is_tensor(t):
        if isinstance(t, (list, tuple)) and len(t) > 0 and torch.is_tensor(t[-1]):
            t = t[-1]
        elif isinstance(t, dict) and len(t) > 0 and torch.is_tensor(list(t.values())[-1]):
            t = list(t.values())[-1]
        else:
            raise RuntimeError("Backbone output type not understood.")
    return t


class BackboneEncoder(nn.Module):
    """Wrap backbone to extract and pool features."""
    def __init__(self, bb):
        super().__init__()
        self.bb = bb
    
    def forward(self, x):
        out = self.bb(x)
        t = _pick_last_feat(out)
        # Shape adaptation
        if t.ndim == 4:
            z = t.mean(dim=(2, 3))  # Global average pooling
        elif t.ndim == 3:
            z = t.mean(dim=2)
        elif t.ndim == 2:
            z = t
        else:
            z = t.view(t.size(0), -1)
        return z


student_enc = BackboneEncoder(student_bb).to(device)
teacher_enc = BackboneEncoder(teacher_bb).to(device)

# Get feature dimension
with torch.no_grad():
    dmy = torch.zeros(1, 3, 224, 224, device=device)
    feat_dim = student_enc(dmy).shape[1]

print(f"✓ Backbone feature dimension: {feat_dim}")

# ===============================================================
# 6) Build YOLOv10s Model and Extract Backbone
# ===============================================================

In [ ]:
class MultiCropDINO(Dataset):
    """
    DINO-style multi-crop dataset for SSL pretraining.
    Generates:
      - 2 global crops (224×224)
      - 8 local crops (96×96, resized to 224×224)
    """
    SUPP = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    
    def __init__(self, roots, g_size=224, l_size=96, n_local=8):
        self.files = [p for r in roots for s in self.SUPP for p in Path(r).rglob(s)]
        if not self.files:
            raise RuntimeError(f"No images found in {roots}")
        
        # Global crop 1 (with strong augmentation)
        self.g1 = transforms.Compose([
            transforms.RandomResizedCrop(g_size, scale=(0.4, 1.0), 
                                        interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomGrayscale(0.2),
            transforms.GaussianBlur(1, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
        ])
        
        # Global crop 2 (with augmentation)
        self.g2 = transforms.Compose([
            transforms.RandomResizedCrop(g_size, scale=(0.4, 1.0), 
                                        interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomGrayscale(0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
        ])
        
        # Local crops (multi-scale, small regions)
        self.local = transforms.Compose([
            transforms.RandomResizedCrop(l_size, scale=(0.05, 0.4), 
                                        interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.Resize(g_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomGrayscale(0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
        ])
        self.n_local = n_local
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        crops = [self.g1(img), self.g2(img)]
        crops.extend([self.local(img) for _ in range(self.n_local)])
        return crops


# Visualize sample crops
print("\n=== VISUALIZING DINO MULTI-CROPS ===")
ds_sample = MultiCropDINO([UNLABELED_DIR], g_size=224, l_size=96, n_local=8)
print(f"Dataset size: {len(ds_sample)} images")

# Denormalize for visualization
def denorm(t):
    t = t.clone()
    t[0] = t[0] * 0.229 + 0.485
    t[1] = t[1] * 0.224 + 0.456
    t[2] = t[2] * 0.225 + 0.406
    return torch.clamp(t, 0, 1)

if len(ds_sample) > 0:
    crops = ds_sample[0]
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for i, crop in enumerate(crops[:10]):
        ax = axes[i // 5, i % 5]
        ax.imshow(denorm(crop).permute(1, 2, 0).numpy())
        ax.set_title(f"Crop {i+1}\n{'Global' if i < 2 else 'Local'}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "01_dino_crops_visualization.png", dpi=100, bbox_inches='tight')
    plt.show()
    print("✓ Crops visualization saved")

# ===============================================================
# 5) Multi-Crop Dataset for DINO (SSL Training)
# ===============================================================

In [ ]:
print("\n=== COPYING IMAGES FOR SSL (UNLABELED DATASET) ===")

# Copy all images to unlabeled directory (for SSL pretraining)
unlabeled_copied = 0
for species in fish_species:
    species_dir = IMAGES_DIR / species
    if species_dir.exists():
        for img_file in species_dir.glob("*.*"):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.gif']:
                dst = UNLABELED_DIR / img_file.name
                if not dst.exists():
                    shutil.copy2(img_file, dst)
                    unlabeled_copied += 1

print(f"✓ Copied {unlabeled_copied} images to {UNLABELED_DIR}")
print(f"  Total unlabeled images available for SSL: {len(list(UNLABELED_DIR.glob('*.*')))}")

# ===============================================================
# 4) Prepare Unlabeled Images Directory (for SSL)
# ===============================================================

In [ ]:
# Explore dataset structure
print("\n=== DATASET EXPLORATION ===\n")

# List fish species (subdirectories)
fish_species = sorted([d.name for d in IMAGES_DIR.iterdir() if d.is_dir()])
print(f"Fish species found: {len(fish_species)}")
for sp in fish_species:
    img_count = len(list((IMAGES_DIR / sp).glob("*.*")))
    print(f"  - {sp}: {img_count} images")

# Count total images
total_images = sum(len(list((IMAGES_DIR / sp).glob("*.*"))) for sp in fish_species)
print(f"\nTotal images: {total_images}")

# Check annotation structure
ann_files = list(ANNOTATIONS_DIR.glob("*.json"))
print(f"\nAnnotation files found: {len(ann_files)}")
for af in ann_files[:5]:
    print(f"  - {af.name}")

# Load and examine first annotation file
if ann_files:
    with open(ann_files[0], 'r') as f:
        sample_ann = json.load(f)
    if isinstance(sample_ann, list):
        print(f"\nFirst annotation file has {len(sample_ann)} entries")
    else:
        print(f"\nFirst annotation structure keys: {list(sample_ann.keys())}")

# ===============================================================
# 3) Explore SylishBD Dataset Structure
# ===============================================================

In [ ]:
# ========== DATASET PATHS (SylishBD) ==========
BASE = Path("/kaggle/input/syfish-bd/Sylfish_bd")
IMAGES_DIR = BASE / "images"
ANNOTATIONS_DIR = BASE / "annotations"
MASKS_DIR = BASE / "masks"

# ========== WORKING DIRECTORIES (Kaggle) ==========
WORK = Path("/kaggle/working/sylfishbd_ssl_dino_yolo")
UNLABELED_DIR = WORK / "unlabeled_sylfishbd"
YOLO_DIR = WORK / "yolo_sylfishbd"
FEATURES_DIR = WORK / "features"
RESULTS_DIR = WORK / "results"

# ========== Model weights paths ==========
SSL_BACKBONE_W = WORK / "backbone_ssl_v10_dinov2.pt"
DATA_YAML = WORK / "data_sylfishbd.yaml"
YOLO_SPLIT = YOLO_DIR / "split"

# Create all working directories
for d in [WORK, UNLABELED_DIR, YOLO_DIR, FEATURES_DIR, RESULTS_DIR, YOLO_SPLIT]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  Base dataset: {BASE}")
print(f"  Working directory: {WORK}")
print(f"  Unlabeled images: {UNLABELED_DIR}")
print(f"  YOLO dataset: {YOLO_DIR}")
print(f"  Features output: {FEATURES_DIR}")
print(f"  Results output: {RESULTS_DIR}")

# ===============================================================
# 2) Configure Paths (Kaggle Dataset & Working Directories)
# ===============================================================

In [ ]:
import os, json, yaml, math, gc, random, shutil, contextlib
from pathlib import Path
from PIL import Image
import cv2

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import pandas as pd

# Configure device and random seeds
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("=" * 70)
print("CSE-475: Semi-Supervised Learning (SSL) with DINO + YOLO")
print("Dataset: SylishBD (Fish Detection)")
print("=" * 70)
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Utility for autocast context
def autocast_ctx():
    return torch.autocast(device_type="cuda", enabled=True) if device == "cuda" else contextlib.nullcontext()

# ===============================================================
# 1) Import Libraries and Configure Environment
# ===============================================================

In [ ]:
!pip -q install torch==2.2.1 torchvision==0.17.1 pytorch-lightning==2.2.1 lightly==1.5.22 ultralytics==8.2.6
!pip -q install opencv-python matplotlib tqdm pycocotools scikit-learn pillow numpy pandas

# ===============================================================
# 0) Setup: Install Required Packages (Kaggle Python 3.11)
# ===============================================================

# ===============================================================
# 📜 CSE-475: Semi-Supervised Learning (SSL) with DINO + YOLO
# ===============================================================
# **SylishBD Fish Detection Dataset**

This notebook implements a **DINOv2-style Self-Supervised Learning (SSL)**
pretraining on the **YOLOv10 backbone**, followed by **YOLO12 detector** 
fine-tuning on the **SylishBD fish detection dataset**.

## 🔹 What's Inside:
   1. **COCO → YOLO conversion** (images/labels + data.yaml)
   2. **SSL pretraining** (DINOv2-style):
       - Student/Teacher (EMA) with cosine momentum schedule
       - Multi-crop views: 2 global + 8 local crops
       - Temperature schedule for teacher + probability centering
       - Cross-entropy between teacher probs (global) and student logits (all views)
   3. **Save SSL-pretrained YOLOv10 backbone weights**
   4. **Fine-tune YOLO12 detector** with SSL backbone initialization
   5. **Evaluate** (mP, mR, mAP@0.50, mAP@0.50–0.95)
   6. **Visualize** backbone features (PCA) and detection predictions

## ⚠️ Dataset: SylishBD
   - **Images**: `/kaggle/input/syfish-bd/Sylfish_bd/images`
   - **Annotations**: `/kaggle/input/syfish-bd/Sylfish_bd/annotations` (COCO JSON)
   - **Masks**: `/kaggle/input/syfish-bd/Sylfish_bd/masks` (optional)
   - **Fish species**: Boal, Ilish, Kalibaush, Katla, Koi, Mrigel, Pabda, Rui, Telapia

## ⚠️ Note:
   - This is a **DINOv2-style adaptation** to convolutional backbones for YOLOv10 compatibility
   - For better transfer learning, increase **SSL_EPOCHS** to 50–100
   - Kaggle-compatible: uses `/kaggle/working/` for outputs

---